In [102]:
import pandas as pd
import numpy as np
import pytz

In [103]:
def remove_unecessary(df, to_remove, name):
    """
    Remove unnecessary columns from the DataFrame.
    """
    df = df.drop(columns=[col for col in to_remove if col in df.columns])
    df.to_csv(f"ShortenedData/{name}.csv", index=False)

def preprocess_QuantAQ(file_path, start_date, end_date):
    sensor = pd.read_csv(file_path, parse_dates=['timestamp'])
    sensor = sensor[['timestamp_local','rh','temp','no2']].dropna(subset=['no2'])
    sensor.rename(columns={'timestamp_local':'time'}, inplace=True)
    
    sensor['time'] = pd.to_datetime(sensor['time'])
    est = pytz.timezone('US/Eastern')
    sensor['time'] = sensor['time'].dt.tz_convert(est)
    sensor['time'] = sensor['time'].dt.strftime('%Y-%m-%d %H:%M:%S')
    sensor['time'] = pd.to_datetime(sensor['time'])
    sensor['day'] = sensor['time'].dt.date
    sensor['dayhour'] = sensor['time'].dt.strftime('%Y-%m-%d %H')
    sensor = sensor[(sensor['time'] >= start_date) & (sensor['time'] <= end_date)]

    sensor = sensor.groupby('dayhour').agg(
        shourlyNO2=('no2', lambda x: x.mean(skipna=True)),
        shourlyRH=('rh', lambda x: x.mean(skipna=True)),
        shourlyTemp=('temp', lambda x: x.mean(skipna=True))
    ).reset_index()

    # sensor = (
    #     sensor.groupby('dayhour', as_index=False)
    #     .agg({
    #         'no2': 'mean',
    #         'rh': 'mean',
    #         'temp': 'mean'
    #     })
    #     .rename(columns={
    #         'no2': 'hourlyNO2',
    #         'rh': 'hourlyRH',
    #         'temp': 'hourlyTemp'
    #     })
    # )

    # Add day column again for convenience
    # hourly['day'] = hourly['dayhour'].dt.date

    # sensor = sensor.groupby('dayhour')['no2'].mean().reset_index()

    sensor.to_csv("testing3.csv", index=False)

In [104]:
# remove = ["bin0","bin1","bin2","bin3","bin4","bin5","bin6","bin7","bin8","bin9",
# "bin10","bin11","bin12","bin13","bin14","bin15","bin16","bin17","bin18","bin19",
# "bin20","bin21","bin22","bin23","opcn3_pm1","opcn3_pm25","opcn3_pm10","pm1_env","pm25_env","pm10_env",
# "neph_bin0","co_we","co_ae","co_diff","no_we","no_ae","no_diff","no2_we","no2_ae","no2_diff",
# "o3_we","o3_ae","ox_diff","flag","lat","lon","device_state","pm1_model_id","pm25_model_id","pm10_model_id",
# "co_model_id","no_model_id","no2_model_id","o3_model_id"]
# df = pd.read_csv("BiasCorrectionScript/data/MOD-00589-Peoplestown Rick McDevitt Youth Center.csv")
# remove_unecessary(df, remove, "MOD-00589-RAW")
# df = pd.read_csv("BiasCorrectionScript/data/MOD-00590-South DeKalb 6.csv")
# remove_unecessary(df, remove, "MOD-00590-RAW")
# df = pd.read_csv("BiasCorrectionScript/data/MOD-00591-Pittsburgh Carey Executive Limousine.csv")
# remove_unecessary(df, remove, "MOD-00591-RAW")
# df = pd.read_csv("BiasCorrectionScript/data/MOD-00592-Chapel Hill Decatur.csv")
# remove_unecessary(df, remove, "MOD-00592-RAW")
# df = pd.read_csv("BiasCorrectionScript/data/MOD-00593-South DeKalb.csv")
# remove_unecessary(df, remove, "MOD-00593-RAW")
# df = pd.read_csv("BiasCorrectionScript/data/MOD-00595-South DeKalb 3.csv")
# remove_unecessary(df, remove, "MOD-00595-RAW")



In [105]:
# df = pd.read_csv("../BiasCorrectionScript/ShortenedData/MOD-00590-RAW.csv")
preprocess_QuantAQ("../BiasCorrectionScript/ShortenedData/MOD-00590-RAW.csv", "2025-05-31", "2025-07-16")